# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their `@id` values for maximal reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the latest mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata as a Dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata (as a single object)
metadata = dataset.metadata
# Print high-level information
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values as defined in the Croissant schema.

In [ ]:
# List all record sets defined in the Croissant metadata

record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets available in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet Name: {rs.name} (@id={rs['@id']})")
        # List the fields and columns
        print(" Fields:")
        for f in rs.field:
            print(f"   - {f.name} (@id={f['@id']}, dtype={getattr(f, 'dataType', 'n/a')})")
        print(" Columns:")
        for c in getattr(rs, 'column', []):
            print(f"   - {c.name} (@id={c['@id']})")
        print()
    # For demonstration, select the first RecordSet for further extraction
    main_record_set_id = record_sets[0]['@id'] if record_sets else None
    print(f"\nMain record set for extraction: {main_record_set_id}")

## 3. Data Extraction
Load data from the primary record set into a DataFrame, referencing entities by their `@id` for clarity and consistency.

In [ ]:
# Extract all records from available record sets
dataframes = {}
record_set_ids = []
if record_sets:
    for rs in record_sets:
        rs_id = rs['@id']
        record_set_ids.append(rs_id)
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

    # Display columns from the main record set
    main_rs_id = record_set_ids[0]
    print(f"Columns in main record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Reference all fields using their `@id` values.

In [ ]:
# Identify a numeric field and group field using @id
if record_sets:
    main_rs_id = record_set_ids[0]
    df = dataframes[main_rs_id].copy()

    # Find a field with numeric dtype
    numeric_field_id = None
    group_field_id = None
    fields = record_sets[0].field
    for f in fields:
        if getattr(f, 'dataType', None) in ['schema:Float', 'schema:Integer', 'Float', 'Integer']:
            numeric_field_id = f['@id']
            numeric_field_name = f.name
            break
    if numeric_field_id is None:
        numeric_field_id = df.select_dtypes(include=['number']).columns[0] if not df.empty else None

    # Choose group field as a categorical variable with multiple values
    for f in fields:
        dt = getattr(f, 'dataType', None)
        if dt == 'Text' or dt == 'schema:Text' or dt is None:
            group_field_id = f['@id']
            group_field_name = f.name
            break
    # Map @id to column name if necessary
    numeric_col = numeric_field_id if numeric_field_id in df.columns else numeric_field_name
    group_col = group_field_id if group_field_id in df.columns else group_field_name

    threshold = 10
    if numeric_col in df.columns:
        filtered_df = df[df[numeric_col] > threshold]
        print(f"Filtered records with {numeric_col} (@id={numeric_field_id}) > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
        print(f"Normalized {numeric_col} for filtered records:")
        display(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

        if group_col in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_col)[numeric_col].mean().reset_index().rename(columns={numeric_col:'mean_'+numeric_col})
            print(f"Grouped data by {group_col} (@id={group_field_id}):")
            display(grouped_df.head())
    else:
        print(f"No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Reference all fields using their `@id`.

In [ ]:
# Create visualizations from the main record set
if record_sets:
    main_rs_id = record_set_ids[0]
    df = dataframes[main_rs_id]

    # Plot histogram for the numeric field
    if numeric_col in df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_col].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_col} (@id={numeric_field_id})")
        plt.xlabel(numeric_col)
        plt.show()

    # Scatter plot (if there are at least two numeric columns)
    numeric_columns = df.select_dtypes(include='number').columns
    if len(numeric_columns) >= 2:
        plt.figure(figsize=(8,5))
        sns.scatterplot(data=df, x=numeric_columns[0], y=numeric_columns[1])
        plt.title(f"Scatterplot of {numeric_columns[0]} vs {numeric_columns[1]}")
        plt.xlabel(numeric_columns[0])
        plt.ylabel(numeric_columns[1])
        plt.show()

    # Bar plot: group counts for group field
    if group_col in df.columns:
        plt.figure(figsize=(8,5))
        grp_counts = df[group_col].value_counts()
        sns.barplot(x=grp_counts.index, y=grp_counts.values)
        plt.title(f"Counts by {group_col} (@id={group_field_id})")
        plt.xlabel(group_col)
        plt.ylabel("Count")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

+ The FAIR^2 dataset metadata, fields, and record sets were loaded directly from the Croissant schema using `mlcroissant`, referencing all entities by their `@id`.
+ Data exploration revealed primary clinical and molecular variables suitable for further study.
+ Common numeric variables were filtered, normalized, and grouped using their `@id`, demonstrating flexible, reproducible data science workflows.
+ Visualizations illustrated data distribution and relationships between fields, further supporting downstream analyses and FAIR research principles.

For model development and additional clinical analysis, reference all fields and record sets by their `@id`.